# Лабораторная работа № 1. Энтропия браузерных метрик

**Цель.** Сравнить информативность шести метрик по эмпирическим энтропиям Шеннона и Реньи.

**Источники.** Шесть файлов `metric_*.csv`, презентации «ПЗ - 1» и «СФП», примеры из «Энтропия.xlsx». Для повторного запуска откройте ноутбук из корня проекта с папкой `data`.

## 1. Частоты и меры неопределённости

Для каждого наблюдаемого значения $i$ частота $n_i$ даёт вероятность $p_i=n_i/N$, где $N$ — объём выборки. Строка CSV представляет одно наблюдение. Например, список языков рассматривается как одно составное значение.

$$
H_1=-\sum_i p_i\log_2 p_i,\qquad
H_2=-\log_2\sum_i p_i^2,\qquad
H_{\max}=\log_2 k.
$$

Здесь $k$ — число различных наблюдаемых значений. $H_2$ — энтропия Реньи порядка 2. Сумма квадратов вероятностей равна вероятности совпадения двух независимых наблюдений из эмпирического распределения. При $k$ равновероятных исходах обе энтропии равны $\log_2 k$.

In [1]:
import csv
from collections import Counter
from math import fsum, isclose, log2
from pathlib import Path

METRICS = {
    'agent': 'metric_1_agent.csv',
    'cpucores': 'metric_2_cpucores.csv',
    'ip': 'metric_3_ip.csv',
    'langsDetected': 'metric_4_langsDetected.csv',
    'language': 'metric_5_language.csv',
    'resolution': 'metric_6_resolution.csv',
}

def load_metric(path):
    with path.open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.reader(stream))
    if not rows or any(len(row) != 1 for row in rows):
        raise ValueError(f'Ожидалась непустая выборка из одного столбца в {path.name}')
    return [row[0] for row in rows]

def entropies(counts):
    total = sum(counts.values())
    if total <= 0:
        raise ValueError('Пустая выборка')
    probabilities = [count / total for count in counts.values() if count > 0]
    shannon = fsum(-p * log2(p) for p in probabilities)
    renyi_2 = -log2(fsum(p ** 2 for p in probabilities))
    return shannon, renyi_2, log2(len(probabilities))

for counts, expected in [
    ({'a': 6, 'b': 4}, 0.9709505944546686),
    ({'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 16}, 1.1219280948873624),
]:
    assert isclose(entropies(counts)[0], expected)
assert entropies({'a': 1}) == (0, 0, 0)
assert entropies({'a': 1, 'b': 1}) == (1, 1, 1)
print('Проверены два примера из таблицы и два граничных случая.')

Проверены два примера из таблицы и два граничных случая.


## 2. Сравнение шести метрик

Для каждой метрики считаются объём, число различных значений, частота наиболее распространённого значения и три меры в битах. Таблица отсортирована по энтропии Шеннона.

In [2]:
directory = Path('data')
results = []
for metric, filename in METRICS.items():
    counts = Counter(load_metric(directory / filename))
    shannon, renyi_2, upper = entropies(counts)
    total = counts.total()
    most_common_share = max(counts.values()) / total
    assert -1e-12 <= renyi_2 <= shannon + 1e-12 <= upper + 2e-12
    results.append((metric, total, len(counts), most_common_share, shannon, renyi_2, upper))

results.sort(key=lambda row: row[4], reverse=True)
print(f'{"Метрика":<15} {"N":>4} {"k":>4} {"max p":>8} {"H₁":>8} {"H₂":>8} {"Hmax":>8}')
for metric, total, distinct, share, shannon, renyi_2, upper in results:
    print(f'{metric:<15} {total:>4} {distinct:>4} {share:>8.3f} {shannon:>8.3f} {renyi_2:>8.3f} {upper:>8.3f}')
print()
print('Лидер по H₁', results[0][0])
print('Лидер по H₂', max(results, key=lambda row: row[5])[0])
print('Минимум H₁', results[-1][0])

Метрика            N    k    max p       H₁       H₂     Hmax
resolution        38   19    0.132    4.008    3.795    4.248
agent             38   18    0.158    3.850    3.565    4.170
ip                38   10    0.263    3.022    2.782    3.322
language          38    9    0.316    2.623    2.277    3.170
cpucores          38    5    0.553    1.683    1.361    2.322
langsDetected     38    4    0.605    1.586    1.253    2.000

Лидер по H₁ resolution
Лидер по H₂ resolution
Минимум H₁ langsDetected


## Вывод

В каждой из шести выборок по 38 наблюдений. Наибольшие значения у `resolution` (4,008 бита по Шеннону и 3,795 бита по Реньи). Наименьшие значения у `langsDetected` (1,586 и 1,253 бита), где один исход встречается примерно в 61% наблюдений. Для вывода об устойчивости метрик и идентификации устройств нужны повторные измерения с метками устройств.